<a href="https://colab.research.google.com/github/Kingelanci/graphysics/blob/main/pali_verification_CORRECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pāli Canon Stylometric Analysis - VERIFICATION NOTEBOOK

**Parametri CORRETTI da METODOLOGIA_FILTRI.md**

---

## TRAINING CORPORA

**EARLY CORE (2,406 segmenti):**
- snp4.* (Aṭṭhakavagga): 893
- snp5.* (Pārāyanavagga): 877
- snp1.12 (Munisutta): 72
- snp1.3 (Khaggavisāṇa): 166
- snp3.11 (Nālakasutta): 178
- snp3.6 (Sabhiyasutta): 220

**LATE CORE (11,212 segmenti):**
- bv (Buddhavaṃsa): 3,882
- pv (Petavatthu): 3,366
- vv (Vimānavatthu): 3,964

## AUC ATTESI
- Segmenti: **0.849 ± 0.006**
- Blocchi: **0.980 ± 0.008**

## FILTRI
- p_early ≥ 0.90
- block_p ≥ 0.80
- sutta_p_mean ≥ 0.60

## RISULTATI ATTESI
- Segmenti filtrati: **4,649 / 301 sutta**
- Blocchi filtrati: **660 / 242 sutta**

In [ ]:
!pip install -q scikit-learn pandas numpy

In [ ]:
import json
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')
print('✓ Imports OK')

---
# PART A: DOWNLOAD E LOAD DATI
---

In [ ]:
!rm -rf bilara-data
!git clone --depth 1 --filter=blob:none --sparse https://github.com/suttacentral/bilara-data.git
%cd bilara-data
!git sparse-checkout set root/pli/ms/sutta
%cd ..
print('✓ Downloaded')

In [ ]:
BILARA_ROOT = Path('bilara-data/root/pli/ms/sutta')
EXCLUDE_COLLECTIONS = {'mnd', 'cnd'}  # Niddesa

def load_sutta_pattern(pattern):
    """Load segments matching a sutta pattern."""
    segments = []

    # Handle chapter patterns like snp4, snp5
    if re.match(r'^snp[0-5]$', pattern):
        base = BILARA_ROOT / 'kn' / 'snp'
        for f in base.glob(f'**/{pattern}.*_root-pli-ms.json'):
            with open(f) as fp:
                data = json.load(fp)
            sutta_id = f.stem.replace('_root-pli-ms', '')
            for seg_id, text in data.items():
                if text and len(str(text).split()) >= 2:
                    segments.append({'segment_id': seg_id, 'sutta_id': sutta_id, 'text': text, 'source': pattern})
    else:
        # Handle individual sutta patterns
        if pattern.startswith('snp'):
            base = BILARA_ROOT / 'kn' / 'snp'
        elif pattern.startswith('dn'):
            base = BILARA_ROOT / 'dn'
        elif pattern.startswith('mn'):
            base = BILARA_ROOT / 'mn'
        elif pattern.startswith('sn'):
            base = BILARA_ROOT / 'sn'
        elif pattern.startswith('an'):
            base = BILARA_ROOT / 'an'
        else:
            base = BILARA_ROOT / 'kn'

        for f in base.glob(f'**/{pattern}_root-pli-ms.json'):
            with open(f) as fp:
                data = json.load(fp)
            sutta_id = f.stem.replace('_root-pli-ms', '')
            for seg_id, text in data.items():
                if text and len(str(text).split()) >= 2:
                    segments.append({'segment_id': seg_id, 'sutta_id': sutta_id, 'text': text, 'source': pattern})

    return segments


def load_collection(coll):
    """Load entire collection (bv, pv, vv)."""
    base = BILARA_ROOT / 'kn' / coll
    if not base.exists():
        return []
    segments = []
    for f in sorted(base.glob('**/*_root-pli-ms.json')):
        sutta_id = f.stem.replace('_root-pli-ms', '')
        with open(f) as fp:
            data = json.load(fp)
        for seg_id, text in data.items():
            if text and len(str(text).split()) >= 2:
                segments.append({'segment_id': seg_id, 'sutta_id': sutta_id, 'text': text, 'source': coll})
    return segments


def load_nikaya(nikaya):
    """Load entire nikaya."""
    base = BILARA_ROOT / nikaya
    segments = []
    for f in sorted(base.glob('**/*_root-pli-ms.json')):
        rel_path = str(f.relative_to(BILARA_ROOT))
        if any(f'/{exc}/' in rel_path for exc in EXCLUDE_COLLECTIONS):
            continue
        sutta_id = f.stem.replace('_root-pli-ms', '')
        with open(f) as fp:
            data = json.load(fp)
        for seg_id, text in data.items():
            if text and len(str(text).split()) >= 2:
                segments.append({
                    'segment_id': seg_id,
                    'sutta_id': sutta_id,
                    'text': text,
                    'file_path': rel_path,
                    'nikaya': nikaya
                })
    return segments

print('✓ Functions ready')

---
# PART B: TRAINING CORPORA
---

In [ ]:
# EARLY CORE - 2,406 segmenti attesi
print('Loading EARLY CORE...')
early_segs = []

early_sources = [
    ('snp4', 893),
    ('snp5', 877),
    ('snp1.12', 72),
    ('snp1.3', 166),
    ('snp3.11', 178),
    ('snp3.6', 220),
]

for pattern, expected in early_sources:
    segs = load_sutta_pattern(pattern)
    status = '✓' if abs(len(segs) - expected) <= 10 else '⚠'
    print(f'  {pattern}: {len(segs)} (atteso: {expected}) {status}')
    early_segs.extend(segs)

print(f'\n→ TOTALE EARLY: {len(early_segs)} (atteso: 2,406)')
N_EARLY = len(early_segs)

In [ ]:
# LATE CORE - 11,212 segmenti attesi
print('Loading LATE CORE...')
late_segs = []

late_sources = [
    ('bv', 3882),
    ('pv', 3366),
    ('vv', 3964),
]

for coll, expected in late_sources:
    segs = load_collection(coll)
    status = '✓' if abs(len(segs) - expected) <= 50 else '⚠'
    print(f'  {coll}: {len(segs)} (atteso: {expected}) {status}')
    late_segs.extend(segs)

print(f'\n→ TOTALE LATE: {len(late_segs)} (atteso: 11,212)')
N_LATE = len(late_segs)

print(f'\n→ TOTALE TRAINING: {N_EARLY + N_LATE} (atteso: 13,618)')
print(f'→ Ratio early:late = 1:{N_LATE/N_EARLY:.1f}')

---
# PART C: FEATURE EXTRACTION E MODELLO
---

In [ ]:
def get_lex_features(texts):
    """Extract TTR and avg word length."""
    out = []
    for t in texts:
        words = str(t).split()
        n = len(words) if words else 1
        ttr = len(set(words)) / n
        avg_len = np.mean([len(w) for w in words]) if words else 0
        out.append([ttr, avg_len])
    return np.array(out, dtype=np.float32)


def train_model(train_texts, train_labels, return_components=False):
    """Train model with CORRECT parameters."""

    # TF-IDF: char n-grams 3-5, max 2000 features, min_df=2
    vectorizer = TfidfVectorizer(
        analyzer='char',
        ngram_range=(3, 5),
        max_features=2000,
        min_df=2,
        dtype=np.float32
    )

    X = vectorizer.fit_transform(train_texts).toarray()
    X_lex = get_lex_features(train_texts)
    X = np.hstack([X, X_lex])

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Logistic Regression: C=1.0, class_weight='balanced'
    model = LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight='balanced'
    )

    # 5-fold Stratified CV
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_scaled, train_labels, cv=cv, scoring='roc_auc')

    model.fit(X_scaled, train_labels)

    if return_components:
        return model, vectorizer, scaler, scores
    return scores.mean(), scores.std()


def score_segments(segments, vectorizer, scaler, model):
    """Score segments with trained model."""
    if not segments:
        return []
    texts = [s['text'] for s in segments]
    X = vectorizer.transform(texts).toarray()
    X_lex = get_lex_features(texts)
    X = np.hstack([X, X_lex])
    X = scaler.transform(X)
    probs = model.predict_proba(X)[:, 1]
    for i, s in enumerate(segments):
        s['p_early'] = float(probs[i])
    return segments

print('✓ Model functions ready')

In [ ]:
# TRAIN SEGMENT MODEL
print('='*70)
print('TRAINING SEGMENT MODEL')
print('='*70)

train_texts = [s['text'] for s in early_segs] + [s['text'] for s in late_segs]
train_labels = [1]*len(early_segs) + [0]*len(late_segs)

print(f'Training: {sum(train_labels)} early + {len(train_labels)-sum(train_labels)} late')

model, vectorizer, scaler, cv_scores = train_model(train_texts, train_labels, return_components=True)

SEGMENT_AUC = cv_scores.mean()
SEGMENT_STD = cv_scores.std()

print(f'\nSEGMENT MODEL CV AUC: {SEGMENT_AUC:.3f} ± {SEGMENT_STD:.3f}')
print(f'(atteso: 0.849 ± 0.006)')

status = '✓' if abs(SEGMENT_AUC - 0.849) < 0.02 else '⚠'
print(f'Status: {status}')

In [ ]:
# TRAIN BLOCK MODEL
print('='*70)
print('TRAINING BLOCK MODEL')
print('='*70)

BLOCK_SIZE = 5
MIN_BLOCK_SEGS = 3

def create_blocks(segments, block_size=5, min_segs=3):
    """Create blocks of consecutive segments."""
    blocks = []
    for i in range(0, len(segments), block_size):
        block_segs = segments[i:i+block_size]
        if len(block_segs) >= min_segs:
            block_text = ' '.join([s['text'] for s in block_segs])
            blocks.append({
                'text': block_text,
                'source': block_segs[0].get('source', 'unknown'),
                'n_segs': len(block_segs)
            })
    return blocks

early_blocks = create_blocks(early_segs, BLOCK_SIZE, MIN_BLOCK_SEGS)
late_blocks = create_blocks(late_segs, BLOCK_SIZE, MIN_BLOCK_SEGS)

print(f'Early blocks: {len(early_blocks)}')
print(f'Late blocks: {len(late_blocks)}')

block_texts = [b['text'] for b in early_blocks] + [b['text'] for b in late_blocks]
block_labels = [1]*len(early_blocks) + [0]*len(late_blocks)

block_model, block_vectorizer, block_scaler, block_cv = train_model(
    block_texts, block_labels, return_components=True
)

BLOCK_AUC = block_cv.mean()
BLOCK_STD = block_cv.std()

print(f'\nBLOCK MODEL CV AUC: {BLOCK_AUC:.3f} ± {BLOCK_STD:.3f}')
print(f'(atteso: 0.980 ± 0.008)')

status = '✓' if abs(BLOCK_AUC - 0.980) < 0.02 else '⚠'
print(f'Status: {status}')

print(f'\nImprovement: +{BLOCK_AUC - SEGMENT_AUC:.3f}')

---
# PART D: SCORE ENTIRE CANON
---

In [ ]:
print('Scoring entire canon...')
all_results = []

expected_counts = {
    'dn': 16212,
    'mn': 26775,
    'sn': 38997,
    'an': 38047,
    'kn': 120322,
}

for nikaya in ['dn', 'mn', 'sn', 'an', 'kn']:
    print(f'\n  Processing {nikaya.upper()}...')
    segments = load_nikaya(nikaya)
    segments = score_segments(segments, vectorizer, scaler, model)

    df = pd.DataFrame(segments)
    df.to_csv(f'{nikaya}_scores.csv', index=False)

    expected = expected_counts[nikaya]
    status = '✓' if abs(len(df) - expected) < 100 else '⚠'
    print(f'    {len(df):,} segments (atteso: {expected:,}) {status}')

    all_results.append(df)

all_data = pd.concat(all_results, ignore_index=True)
print(f'\n→ TOTALE CANONE: {len(all_data):,} (atteso: 240,353)')

---
# PART E: GREY ZONE E BLOCCHI
---

In [ ]:
# GREY ZONE: Escludi training
print('='*70)
print('GREY ZONE')
print('='*70)

# Pattern CORRETTO per escludere training
TRAIN_PATTERN = r'^(snp4\.|snp5\.|snp1\.12$|snp1\.3$|snp3\.11$|snp3\.6$|bv|pv|vv)'

grey = all_data[~all_data['sutta_id'].str.match(TRAIN_PATTERN, na=False)].copy()

print(f'Canone totale: {len(all_data):,}')
print(f'Training escluso: {len(all_data) - len(grey):,}')
print(f'Grey zone: {len(grey):,} (atteso: 226,735)')

In [ ]:
# CALCOLO BLOCCHI
print('\n' + '='*70)
print('CALCOLO BLOCCHI')
print('='*70)

blocks_list = []
seg_to_block = {}

for sutta_id in grey['sutta_id'].unique():
    sutta_df = grey[grey['sutta_id'] == sutta_id].sort_values('segment_id')
    nikaya = sutta_df['nikaya'].iloc[0]
    segs = sutta_df.to_dict('records')

    for i in range(0, len(segs), BLOCK_SIZE):
        block_segs = segs[i:i+BLOCK_SIZE]
        if len(block_segs) >= MIN_BLOCK_SEGS:
            block_p = np.mean([s['p_early'] for s in block_segs])
            seg_ids = [s['segment_id'] for s in block_segs]

            blocks_list.append({
                'sutta_id': sutta_id,
                'nikaya': nikaya,
                'block_idx': i // BLOCK_SIZE,
                'block_p': block_p,
                'n_segs': len(block_segs),
                'segment_ids': '|'.join(seg_ids),
                'text': ' '.join([s['text'] for s in block_segs])[:500]
            })

            for seg_id in seg_ids:
                seg_to_block[seg_id] = block_p

blocks_df = pd.DataFrame(blocks_list)
print(f'Blocchi totali: {len(blocks_df):,} (atteso: 45,357)')

# Aggiungi block_p ai segmenti
grey['block_p'] = grey['segment_id'].map(seg_to_block)

# Calcola sutta_p_mean
sutta_block_means = blocks_df.groupby('sutta_id')['block_p'].mean()
grey['sutta_p_mean'] = grey['sutta_id'].map(sutta_block_means)
blocks_df['sutta_p_mean'] = blocks_df['sutta_id'].map(sutta_block_means)

---
# PART F: FILTRO MULTI-LIVELLO
---

In [ ]:
print('='*70)
print('FILTRO MULTI-LIVELLO')
print('='*70)

# Soglie CORRETTE
SEG_TH = 0.90
BLOCK_TH = 0.80
SUTTA_TH = 0.60

print(f'Soglie: p_early ≥ {SEG_TH}, block_p ≥ {BLOCK_TH}, sutta_p_mean ≥ {SUTTA_TH}')

# Filtro segmenti
filt_segs = grey[
    (grey['p_early'] >= SEG_TH) &
    (grey['block_p'] >= BLOCK_TH) &
    (grey['sutta_p_mean'] >= SUTTA_TH)
].copy()

n_filt_segs = len(filt_segs)
n_filt_suttas = filt_segs['sutta_id'].nunique()

print(f'\nSEGMENTI FILTRATI: {n_filt_segs:,} (atteso: 4,649)')
print(f'SUTTA UNICI: {n_filt_suttas} (atteso: 301)')

seg_status = '✓' if abs(n_filt_segs - 4649) < 200 else '⚠'
sutta_status = '✓' if abs(n_filt_suttas - 301) < 20 else '⚠'
print(f'Status: segmenti {seg_status}, sutta {sutta_status}')

# Per nikaya
print('\nPer nikāya:')
expected_nik = {'dn': (601, 9), 'mn': (1288, 37), 'sn': (1097, 116), 'an': (1000, 99), 'kn': (663, 40)}
for nik in ['dn', 'mn', 'sn', 'an', 'kn']:
    nik_df = filt_segs[filt_segs['nikaya'] == nik]
    exp_seg, exp_sut = expected_nik[nik]
    print(f'  {nik.upper()}: {len(nik_df):,} seg / {nik_df["sutta_id"].nunique()} sutta (atteso: {exp_seg}/{exp_sut})')

In [ ]:
# Filtro blocchi (soglia più alta: block_p ≥ 0.90)
print('\n' + '='*70)
print('FILTRO BLOCCHI')
print('='*70)

BLOCK_FILT_TH = 0.90  # Soglia più alta per blocchi

filt_blocks = blocks_df[
    (blocks_df['block_p'] >= BLOCK_FILT_TH) &
    (blocks_df['sutta_p_mean'] >= SUTTA_TH)
].copy()

n_filt_blocks = len(filt_blocks)
n_filt_block_suttas = filt_blocks['sutta_id'].nunique()

print(f'Soglie: block_p ≥ {BLOCK_FILT_TH}, sutta_p_mean ≥ {SUTTA_TH}')
print(f'\nBLOCCHI FILTRATI: {n_filt_blocks} (atteso: 660)')
print(f'SUTTA UNICI: {n_filt_block_suttas} (atteso: 242)')

block_status = '✓' if abs(n_filt_blocks - 660) < 50 else '⚠'
print(f'Status: {block_status}')

In [ ]:
# Salva file
filt_segs.to_csv('FILTERED_SEGMENTS_FINAL.csv', index=False)
filt_blocks.to_csv('FILTERED_BLOCKS_FINAL.csv', index=False)
print('✓ Saved FILTERED_SEGMENTS_FINAL.csv')
print('✓ Saved FILTERED_BLOCKS_FINAL.csv')

---
# PART G: VERIFICA NUMERI DOTTRINALI
---

In [ ]:
def find_term(pattern, data=None, regex=True):
    """Find segments matching pattern."""
    if data is None:
        data = all_data
    if regex:
        return data[data['text'].str.contains(pattern, case=False, na=False, regex=True)]
    else:
        return data[data['text'].str.contains(pattern, case=False, na=False)]

def report(name, pattern, expected_n=None, expected_p=None, regex=True):
    """Report stats for a term."""
    matches = find_term(pattern, regex=regex)
    n = len(matches)
    p = matches['p_early'].mean() if n > 0 else None

    n_status = ''
    p_status = ''
    if expected_n is not None:
        n_status = '✓' if abs(n - expected_n) <= max(expected_n * 0.1, 5) else '⚠'
    if expected_p is not None and p is not None:
        p_status = '✓' if abs(p - expected_p) <= 0.05 else '⚠'

    print(f'{name:<40} N={n:>5} {n_status}  P={p:.3f if p else 0:.3f} {p_status}')
    return n, p

print('✓ Report functions ready')

In [ ]:
print('='*70)
print('BRAHMAVIHĀRA BIMODAL (Δ = 0.854)')
print('='*70)

report('mettāsahagatena cetasā', 'mettāsahagatena cetasā', 48, 0.956, regex=False)
report('upekkhāsahagatena cetasā', 'upekkhāsahagatena cetasā', 46, 0.960, regex=False)
report('karuṇāsahagatena cetasā', 'karuṇāsahagatena cetasā', 35, 0.102, regex=False)
report('muditāsahagatena cetasā', 'muditāsahagatena cetasā', 34, 0.035, regex=False)

In [ ]:
print('\n' + '='*70)
print('PABHASSARA BIMODAL (Δ = 0.979)')
print('='*70)

report('Goldsmith metaphor', r'mudu.*kammaniya.*pabhassara', 11, 0.979)
report('Pabhassaramidaṁ formula', 'pabhassaramidaṁ', None, 0.000, regex=False)

In [ ]:
print('\n' + '='*70)
print('SAMMĀDIṬṬHI BIMODAL (Δ = 0.777)')
print('='*70)

report('sammādiṭṭhi + pajānāti', r'sammādiṭṭhi.*pajānāti|pajānāti.*sammādiṭṭhi', 1, 0.999)
report('diṭṭhisampanno + aṭṭhāna', r'diṭṭhisampanno.*aṭṭhāna|aṭṭhāna.*diṭṭhisampanno', 4, 0.999)
report('katamā sammādiṭṭhi', 'katamā.*sammādiṭṭhi', 7, 0.217)

In [ ]:
print('\n' + '='*70)
print('EIGHTFOLD PATH (Δ = 0.452)')
print('='*70)

report('sammāsati', 'sammāsati', None, 0.881, regex=False)
report('sammākammanta', 'sammākammanta', None, 0.792, regex=False)
report('sammādiṭṭhi', 'sammādiṭṭhi', None, 0.703, regex=False)
report('sammāvāyāma', 'sammāvāyāma', None, 0.689, regex=False)
report('sammāvācā', 'sammāvācā', None, 0.625, regex=False)
report('sammāājīva', 'sammāājīva', None, 0.596, regex=False)
report('sammāsamādhi', 'sammāsamādhi', None, 0.584, regex=False)
report('sammāsaṅkappa', 'sammāsaṅkappa', None, 0.577, regex=False)
report('ariyo aṭṭhaṅgiko maggo (FORMULA)', 'ariyo aṭṭhaṅgiko maggo', 231, 0.409, regex=False)

In [ ]:
print('\n' + '='*70)
print('TILAKKHAṆA SEQUENCE')
print('='*70)

print('\n-ato anupassati forms:')
report('aniccato anupassati', 'aniccato anupassati', None, 0.955, regex=False)
report('dukkhato anupassati', 'dukkhato anupassati', None, 0.892, regex=False)
report('anattato anupassati', 'anattato anupassati', None, 0.669, regex=False)

print('\nsabbe X forms:')
report('sabbe saṅkhārā aniccā', 'sabbe saṅkhārā aniccā', None, 0.968, regex=False)
report('sabbe saṅkhārā dukkhā', 'sabbe saṅkhārā dukkhā', None, 0.769, regex=False)
report('sabbe dhammā anattā', 'sabbe dhammā anattā', None, 0.695, regex=False)

In [ ]:
print('\n' + '='*70)
print('JHĀNA DESCRIPTION vs NUMERATION')
print('='*70)

report('savitakkaṁ savicāraṁ', 'savitakkaṁ savicāraṁ', 101, 1.000, regex=False)
report('upekkhako satimā', 'upekkhako satimā', 51, 1.000, regex=False)
report('paṭhamaṁ jhānaṁ', 'paṭhamaṁ jhānaṁ', None, None, regex=False)
report('catutthaṁ jhānaṁ', 'catutthaṁ jhānaṁ', None, None, regex=False)

In [ ]:
print('\n' + '='*70)
print('METTĀ AS LIBERATION PATH')
print('='*70)

report('sabbāvantaṁ lokaṁ', 'sabbāvantaṁ lokaṁ', 170, 1.000, regex=False)
report('mettāvihārī', 'mettāvihārī', 8, 0.999, regex=False)
report('vipulena mahaggatena', 'vipulena mahaggatena', 170, 0.986, regex=False)

In [ ]:
print('\n' + '='*70)
print('SYSTEMATIC FORMULAS (ALL LATE)')
print('='*70)

report('avijjāpaccayā saṅkhārā', 'avijjāpaccayā saṅkhārā', None, 0.377, regex=False)
report('kāye kāyānupassī', 'kāye kāyānupassī', None, 0.370, regex=False)
report('cattāro iddhipādā', 'cattāro iddhipādā', None, 0.415, regex=False)
report('cattāri ariyasaccāni', 'cattāri ariyasaccāni', None, 0.431, regex=False)

---
# PART H: LATE MARKERS TEST
---

In [ ]:
print('='*70)
print('LATE MARKERS IN FILTERED SET')
print('='*70)

late_markers = [
    ('cakkavatti', 'cakkavatt'),
    ('7 Buddhas', 'vipassi|sikhi|vessabhu|kakusandha|konagamana|kassapa'),
    ('mahāpurisalakkhaṇa', 'mahāpurisalakkhaṇ'),
    ('lokadhātu', 'lokadhātu'),
    ('thūpa/cetiya', 'thūp|cetiy'),
    ('buddhānussati', 'buddhānussati'),
    ('pāramī', 'pāramī'),
    ('bodhisatta', 'bodhisatt'),
]

print(f'{"Marker":<25} {"N canon":>10} {"P canon":>10} {"N filtered":>12}')
print('-'*60)

for name, pattern in late_markers:
    canon_matches = find_term(pattern, all_data)
    filt_matches = find_term(pattern, filt_segs)

    n_canon = len(canon_matches)
    p_canon = canon_matches['p_early'].mean() if n_canon > 0 else 0
    n_filt = len(filt_matches)

    status = '✓' if n_filt <= 5 else '⚠'
    print(f'{name:<25} {n_canon:>10} {p_canon:>10.3f} {n_filt:>12} {status}')

    if n_filt > 0:
        print(f'  → Sutta: {list(filt_matches["sutta_id"].unique())}')

---
# PART I: FINAL SUMMARY
---

In [ ]:
print('='*70)
print('FINAL SUMMARY')
print('='*70)

print(f'''
DATASET
  Canone totale:     {len(all_data):,} segmenti
  Training Early:    {N_EARLY:,} segmenti
  Training Late:     {N_LATE:,} segmenti
  Grey zone:         {len(grey):,} segmenti

MODELLO
  Segment AUC:       {SEGMENT_AUC:.3f} ± {SEGMENT_STD:.3f} (atteso: 0.849 ± 0.006)
  Block AUC:         {BLOCK_AUC:.3f} ± {BLOCK_STD:.3f} (atteso: 0.980 ± 0.008)

FILTRO
  Segmenti:          {n_filt_segs:,} / {n_filt_suttas} sutta (atteso: 4,649 / 301)
  Blocchi:           {n_filt_blocks} / {n_filt_block_suttas} sutta (atteso: 660 / 242)
  P mean filtrati:   {filt_segs["p_early"].mean():.3f}

FILES SALVATI
  - dn_scores.csv, mn_scores.csv, sn_scores.csv, an_scores.csv, kn_scores.csv
  - FILTERED_SEGMENTS_FINAL.csv
  - FILTERED_BLOCKS_FINAL.csv
''')

In [ ]:
from google.colab import files

download_files = [
    'FILTERED_SEGMENTS_FINAL.csv',
    'FILTERED_BLOCKS_FINAL.csv',
    'dn_scores.csv',
    'mn_scores.csv',
    'sn_scores.csv',
    'an_scores.csv',
    'kn_scores.csv',
]

for f in download_files:
    if os.path.exists(f):
        print(f'Downloading {f}...')
        files.download(f)